# TFIDF — AG News headline classification
**ARTI-404 · Group 5 · Abdulaziz**

Traditional NLP approach: **TF-IDF + Logistic Regression**, using the shared four-class AG News headline splits. The goal is to report the traditional baseline on all test headlines and on the **6-words-or-fewer** subset, then later compare fairly with the pretrained approach.

**Academic note:** the final test was evaluated only after the best settings had already been selected using validation. Do not change parameters based on the final test scores.

## 1. Setup
Install with `python -m pip install -r requirements.txt` in the same folder, and use the provided saved model for quick reproducible execution. Change `RUN_TUNING = True` only to independently repeat all six validation experiments.

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd
from IPython.display import display
from traditional_model import (
    load_shared_data, train_and_select, save_outputs,
    predict_frame, predict_text, export_frozen_test,
)
DATA_DIR = Path("data")
MODEL_PATH = Path("outputs/tfidf/tfidf_logreg.joblib")
OUTPUT_DIR = Path("outputs/tfidf")
RUN_TUNING = False  # True: reproduce the original six-setting search (slow)
print("TFIDF setup ready. RUN_TUNING:", RUN_TUNING)

TFIDF setup ready. RUN_TUNING: False


## 2. Data
The prepared CSVs were produced by Ibrahim from [AG News](https://huggingface.co/datasets/sh0416/ag_news). `text` contains the headline only. Labels: 1 World, 2 Sports, 3 Business, 4 Sci/Tech. `is_short` corresponds to `word_count <= 6`. Training, validation and test splits must be disjoint.

In [2]:
train_df, val_df, test_df = load_shared_data(DATA_DIR, include_test=True)
print(f"Train={len(train_df):,}, Validation={len(val_df):,}, Test={len(test_df):,}")
print("Train labels:")
display(train_df["label"].value_counts().sort_index().rename("count").to_frame())
print("Short test headlines:", int(test_df["is_short"].sum()))

Train=101,043, Validation=11,227, Test=7,464
Train labels:


,count
label,
1,25699
2,26028
3,24922
4,24394


Short test headlines: 3554


## 3. Method — train and select by validation only
The implementation in `traditional_model.py` fits a `TfidfVectorizer` and one-vs-rest `LogisticRegression` inside one scikit-learn `Pipeline`, preventing the vectorizer from learning from validation or test texts. Six combinations were tested: `ngram_range` = (1,1) or (1,2); `C` = 0.5, 2.0, or 8.0. The chosen setting is **(1,1), C=8.0**, selected by validation macro F1.

By default this notebook reuses the previously selected saved model to reproduce the final outputs without repeating training.

In [3]:
if RUN_TUNING:
    model, leaderboard, validation_predictions, validation_metrics, settings = train_and_select(train_df, val_df, quick=False)
    save_outputs(OUTPUT_DIR, model, leaderboard, validation_predictions, validation_metrics, settings, mode="shared")
else:
    if not MODEL_PATH.exists():
        raise FileNotFoundError("Saved model missing. Set RUN_TUNING=True and run training first.")
    model = joblib.load(MODEL_PATH)  # Trusted project artifact only.
    leaderboard = pd.read_csv(OUTPUT_DIR / "hyperparameter_comparison.csv")
    settings = json.loads((OUTPUT_DIR / "model_info.json").read_text(encoding="utf-8"))["selected_settings"]
    validation_predictions, validation_metrics = predict_frame(model, val_df)
print("Validation-selected parameters:", settings)
display(pd.DataFrame(leaderboard).sort_values("macro_f1", ascending=False))

Validation-selected parameters: {'ngram_range': [1, 1], 'C': 8.0}


,ngram_range,C,accuracy,macro_f1
0,"(1, 1)",8.0,0.878507,0.877722
1,"(1, 2)",8.0,0.876993,0.876115
2,"(1, 1)",2.0,0.876726,0.875735
3,"(1, 2)",2.0,0.875212,0.874175
4,"(1, 1)",0.5,0.862831,0.861602
5,"(1, 2)",0.5,0.860693,0.859394


## 4. Validation performance
Accuracy and macro F1 are reported for all validation headlines and for the subset with six words or fewer. These numbers **must not** be confused with final test results.

In [4]:
print("Validation, all headlines:", validation_metrics["overall"])
print("Validation, short headlines:", validation_metrics["short_6_or_less"])
print("Number of short validation headlines:", validation_metrics["short_samples"])

Validation, all headlines: {'accuracy': 0.8785071702146611, 'macro_f1': 0.8777219035122683}
Validation, short headlines: {'accuracy': 0.8662947287418238, 'macro_f1': 0.8642477522348154}
Number of short validation headlines: 5198


## 5. Final test (frozen model; no hyperparameter tuning)
This cell only generates fixed-model predictions, metrics and the file **`results/tfidf_predictions.csv`**. It preserves the exact order and labels of `data/test.csv`, as required by Ibrahim's evaluation code. Do not make model changes in response to these results.

In [5]:
test_metrics = export_frozen_test(
    data_dir=DATA_DIR, model_path=MODEL_PATH,
    output_dir=OUTPUT_DIR, results_dir=Path("results")
)
print("FINAL TEST — overall:", test_metrics["overall"])
print("FINAL TEST — short headlines:", test_metrics["short_6_or_less"])
print("Short test n:", test_metrics["short_samples"])
team_predictions = pd.read_csv("results/tfidf_predictions.csv")
assert team_predictions["text"].equals(test_df["text"])
assert len(team_predictions) == len(test_df)
print("Ibrahim-compatible export verified:", len(team_predictions), "rows")

Frozen model: outputs/tfidf/tfidf_logreg.joblib
Validated 101,043 training rows, 11,227 validation rows, 7,464 test rows
TEST (one-time evaluation; do not retune): {
  "overall": {
    "accuracy": 0.8829046087888531,
    "macro_f1": 0.8822607837461339
  },
  "samples": 7464,
  "short_6_or_less": {
    "accuracy": 0.867191896454699,
    "macro_f1": 0.8655604500754228
  },
  "short_samples": 3554
}
Team prediction file: results/tfidf_predictions.csv
FINAL TEST — overall: {'accuracy': 0.8829046087888531, 'macro_f1': 0.8822607837461339}
FINAL TEST — short headlines: {'accuracy': 0.867191896454699, 'macro_f1': 0.8655604500754228}
Short test n: 3554
Ibrahim-compatible export verified: 7464 rows


## 6. Error analysis
These are actual errors from the held-out test set. A headline does not always provide enough context to identify the article's topic; explanations should be phrased as hypotheses, not as facts about unseen article text.

In [6]:
errors = pd.read_csv("results/tfidf_error_examples.csv")
display(errors[["title", "true_label", "predicted_label", "short_6_or_less"]].head(5))

,title,true_label,predicted_label,short_6_or_less
0,Prediction Unit Helps Forecast Wildfires (AP),Sci/Tech,Sports,True
1,Loosing the War on Terrorism,Sci/Tech,World,True
2,"Card fraud unit nets 36,000 cards",Sci/Tech,Business,True
3,Super ant colony hits Australia,Sci/Tech,World,True
4,IBM to hire even more new workers,Sci/Tech,Business,False


## 7. Try it — new, unseen headline
This predicts using the already-fitted pipeline without retraining; whitespace-only input displays a useful message.

In [7]:
headline = "Team wins the football final"
print("Headline:", headline)
print("Prediction:", predict_text(model, headline))
print("Empty input:", predict_text(model, "   "))

Headline: Team wins the football final
Prediction: Sports
Empty input: Please enter a non-empty English news headline.


## 8. Sources and limitations
- Dataset: [AG News](https://huggingface.co/datasets/sh0416/ag_news)
- Methods: [TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- Limitations: lexical ambiguity, unseen wording, and insufficient context in short headlines.
- **Team pending:** final comparison with the pretrained approach, which needs Mohammed's prediction CSV on the same test split.
- **Student pending:** write two personally created labelled headline examples and retain dated contribution evidence. Credit AI assistance in the final report where used.